In [ ]:
import pandas as pd
import boto3
import pymysql
import os
import time

s3 = boto3.client("s3")

def lambda_handler(event, context):
    start_time = time.time()
    connection = None
    cursor = None
    
    def log_progress(step, elapsed, remaining_time):
        print(f"⏱️ {step}: {elapsed:.2f}s elapsed, {remaining_time}ms remaining")
    
    try:
        bucket = event["Records"][0]["s3"]["bucket"]["name"]
        key = event["Records"][0]["s3"]["object"]["key"]
        
        print(f"🚀 Starting: {key}")
        print(f"💾 Memory: 750MB allocated")
        print(f"⏰ Initial time: {context.get_remaining_time_in_millis()}ms")
        
        # Step 1: Read CSV
        step_start = time.time()
        obj = s3.get_object(Bucket=bucket, Key=key)
        df = pd.read_csv(obj["Body"])
        step_time = time.time() - step_start
        remaining = context.get_remaining_time_in_millis()
        
        print(f"✅ CSV loaded: {len(df)} rows in {step_time:.2f}s")
        log_progress("CSV Read", time.time() - start_time, remaining)
        
        # Step 2: Database connection - FIXED
        step_start = time.time()
        connection = pymysql.connect(
            host=os.environ["DB_HOST"],        # ✅ Fixed: Use simple variable name
            user=os.environ["DB_USER"],        # ✅ Fixed: Use simple variable name
            password=os.environ["DB_PASSWORD"], # ✅ Fixed: Use simple variable name
            database=os.environ["DB_NAME"],     # ✅ Fixed: Use simple variable name
            port=3306,
            connect_timeout=10,
            read_timeout=30,
            write_timeout=30
        )
        cursor = connection.cursor()
        step_time = time.time() - step_start
        remaining = context.get_remaining_time_in_millis()
        
        print(f"✅ DB connected in {step_time:.2f}s")
        log_progress("DB Connection", time.time() - start_time, remaining)
        
        # Step 3: Process data
        batch_size = 50  # Smaller batches for faster feedback
        total_rows = len(df)
        processed = 0
        
        insert_query = """
        INSERT IGNORE INTO transactions (
            transaction_id, merchant, amount, currency,
            payment_method, status, transaction_time
        ) VALUES (%s,%s,%s,%s,%s,%s,%s)
        """
        
        print(f"📊 Processing {total_rows} rows in batches of {batch_size}")
        
        for i in range(0, total_rows, batch_size):
            batch_start = time.time()
            remaining = context.get_remaining_time_in_millis()
            
            # Safety check - stop if running out of time
            if remaining < 60000:  # Less than 1 minute
                print(f"⚠️ STOPPING: Only {remaining}ms remaining")
                break
            
            # Prepare batch
            batch_df = df.iloc[i:i+batch_size]
            records = []
            
            for _, row in batch_df.iterrows():
                records.append((
                    str(row["transaction_id"]), str(row["merchant"]), 
                    float(row["amount"]), str(row["currency"]),
                    str(row["payment_method"]), str(row["status"]), 
                    str(row["transaction_time"])
                ))
            
            # Database insert
            insert_start = time.time()
            cursor.executemany(insert_query, records)
            connection.commit()
            insert_time = time.time() - insert_start
            
            processed += len(records)
            batch_total_time = time.time() - batch_start
            
            # Progress report every 10 batches or if slow
            if (i // batch_size) % 10 == 0 or batch_total_time > 5:
                print(f"📈 Batch {i//batch_size + 1}: {len(records)} rows")
                print(f"   Insert time: {insert_time:.2f}s, Total batch: {batch_total_time:.2f}s")
                print(f"   Progress: {processed}/{total_rows} ({processed/total_rows*100:.1f}%)")
                print(f"   Remaining: {remaining}ms")
        
        total_time = time.time() - start_time
        final_remaining = context.get_remaining_time_in_millis()
        
        print(f"🎉 COMPLETED!")
        print(f"   Total time: {total_time:.2f}s")
        print(f"   Rows processed: {processed}/{total_rows}")
        print(f"   Time remaining: {final_remaining}ms")
        
        return {
            "statusCode": 200,
            "file": key,
            "rows_processed": processed,
            "total_rows": total_rows,
            "execution_time": total_time,
            "success": processed == total_rows
        }
        
    except Exception as e:
        elapsed = time.time() - start_time
        remaining = context.get_remaining_time_in_millis()
        print(f"❌ ERROR after {elapsed:.2f}s: {str(e)}")
        print(f"   Remaining time: {remaining}ms")
        
        if connection:
            try:
                connection.rollback()
            except:
                pass
        raise e
        
    finally:
        if cursor:
            cursor.close()
        if connection:
            connection.close()
